In [41]:
# Cell 1: tools

from copy import deepcopy
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bloqade.analog import start as bloqade_start

# Rb-87 Rydberg interaction coefficient used by Bloqade/Aquila-style examples.
# Units: rad * um^6 / us.
# V(R) = C6 / R^6
RB87_C6_RAD_UM6_PER_US = 2.0 * np.pi * 862690.0

def build_two_atom_program(
    spacing_um: float,
    rabi_amplitude: float,
    detuning: float,
    duration: float,
):
    """
    Build a two-atom Bloqade Analog Rydberg program.

    Geometry:
        atom 0 at (0, 0)
        atom 1 at (spacing_um, 0)

    Program:
        uniform Rabi amplitude
        uniform detuning
        constant waveform duration

    Notes:
        Uses deepcopy(bloqade_start) to avoid accidental accumulation of
        positions across repeated notebook runs.
    """
    geometry = deepcopy(bloqade_start).add_position(
        [(0.0, 0.0), (float(spacing_um), 0.0)]
    )

    program = (
        geometry
        .rydberg
        .rabi.amplitude.uniform.constant(float(rabi_amplitude), float(duration))
        .detuning.uniform.constant(float(detuning), float(duration))
    )

    return program


def get_rydberg_interaction_strength(spacing_um: float) -> float:
    """
    Return the two-atom Rydberg interaction strength V(R) = C6 / R^6.

    Constant:
        RB87_C6_RAD_UM6_PER_US = 2*pi*862690 rad * um^6 / us

    Returned value:
        interaction angular frequency in rad/us

    Notes:
        Dividing by 2*pi gives cycles/us, numerically equivalent to MHz.
    """
    spacing_um = float(spacing_um)

    if spacing_um <= 0:
        return np.nan

    return RB87_C6_RAD_UM6_PER_US / spacing_um**6

def interaction_rad_per_us_to_mhz(interaction_rad_per_us: float) -> float:
    """
    Convert angular frequency from rad/us to MHz.

    Since 1/us = 1 MHz in cycles/us:

        V/2π in MHz = V(rad/us) / (2π)
    """
    return float(interaction_rad_per_us) / (2.0 * np.pi)

def run_two_atom_program(
    spacing_um: float,
    rabi_amplitude: float,
    detuning: float,
    duration: float,
    shots: int,
):
    """
    Build and run a two-atom Bloqade Analog program using the local Python emulator.
    """
    program = build_two_atom_program(
        spacing_um=spacing_um,
        rabi_amplitude=rabi_amplitude,
        detuning=detuning,
        duration=duration,
    )

    batch = program.bloqade.python().run(shots=int(shots))
    report = batch.report()

    return batch, report


def extract_counts_from_report(report) -> OrderedDict:
    """
    Extract the first counts dictionary from a Bloqade report.

    report.counts() returns a list because Bloqade can represent batches.
    This notebook runs one program at a time, so we use the first entry.
    """
    counts_list = report.counts()

    if len(counts_list) == 0:
        raise ValueError("Bloqade report returned no counts.")

    return counts_list[0]


def normalize_bitstring_key(key) -> str:
    """
    Convert Bloqade count keys to compact two-character strings.

    This handles common possibilities such as:
        '01'
        '[0, 1]'
        (0, 1)
    """
    if isinstance(key, str):
        cleaned = (
            key.replace("[", "")
               .replace("]", "")
               .replace(",", "")
               .replace(" ", "")
        )
        return cleaned

    if isinstance(key, (tuple, list, np.ndarray)):
        return "".join(str(int(x)) for x in key)

    return str(key)


def probabilities_from_report(report) -> dict:
    """
    Convert Bloqade report counts into probabilities for a two-atom system.

    Bloqade post-sequence convention used here:
        1 = atom remains in ground/not-Rydberg state
        0 = atom measured in Rydberg excitation state

    Therefore:
        '11' -> gg
        '10' -> gr
        '01' -> rg
        '00' -> rr
    """
    raw_counts = extract_counts_from_report(report)

    canonical_counts = {
        "00": 0,
        "01": 0,
        "10": 0,
        "11": 0,
    }

    for key, value in raw_counts.items():
        clean_key = normalize_bitstring_key(key)
        if clean_key in canonical_counts:
            canonical_counts[clean_key] += int(value)

    total = sum(canonical_counts.values())

    if total == 0:
        raise ValueError("No recognized two-atom bitstrings found in Bloqade counts.")

    probabilities = {
        "rr": canonical_counts["00"] / total,
        "rg": canonical_counts["01"] / total,
        "gr": canonical_counts["10"] / total,
        "gg": canonical_counts["11"] / total,
    }

    return probabilities


def run_spacing_sweep(
    spacings_um,
    rabi_amplitude: float,
    detuning: float,
    duration: float,
    shots: int,
) -> pd.DataFrame:
    """
    Run a spacing sweep and return a dataframe with interaction and probabilities.
    """
    rows = []

    for spacing_um in spacings_um:
        _, report = run_two_atom_program(
            spacing_um=spacing_um,
            rabi_amplitude=rabi_amplitude,
            detuning=detuning,
            duration=duration,
            shots=shots,
        )

        probs = probabilities_from_report(report)
        #interaction = get_rydberg_interaction_strength(spacing_um)
        interaction_rad_per_us = get_rydberg_interaction_strength(spacing_um)
        interaction_mhz = interaction_rad_per_us_to_mhz(interaction_rad_per_us)

        rows.append(
            {
                "spacing_um": float(spacing_um),
                #"interaction": interaction,
                "interaction_rad_per_us": interaction_rad_per_us,
                "interaction_mhz": interaction_mhz,
                "P_rr": probs["rr"],
                "P_rg": probs["rg"],
                "P_gr": probs["gr"],
                "P_gg": probs["gg"],
                "P_single": probs["rg"] + probs["gr"],
                "blockade_suppression": 1.0 - probs["rr"],
            }
        )

    return pd.DataFrame(rows)


def select_nearest_spacing_row(df: pd.DataFrame, selected_spacing_um: float) -> pd.Series:
    """
    Return the dataframe row closest to the requested spacing.
    """
    idx = (df["spacing_um"] - selected_spacing_um).abs().idxmin()
    return df.loc[idx]

def print_compact_summary(
    df: pd.DataFrame,
    selected_row: pd.Series,
    rabi_amplitude: float,
    detuning: float,
    duration: float,
    shots: int,
) -> None:
    """
    Print a compact two-column numerical summary for the notebook.

    Left column:
        global run settings and double-excitation summary

    Right column:
        selected-spacing probability distribution
    """
    min_rr_row = df.loc[df["P_rr"].idxmin()]
    max_rr_row = df.loc[df["P_rr"].idxmax()]

    left_lines = [
        "Bloqade Analog Rydberg Blockade Demo",
        "-" * 44,
        f"Rabi amplitude       : {rabi_amplitude}",
        f"Detuning             : {detuning}",
        f"Pulse duration       : {duration}",
        f"Shots per spacing    : {shots}",
        f"Spacing range        : {df['spacing_um'].min():.3f} to {df['spacing_um'].max():.3f} um",
        "",
        "Double-excitation summary",
        "-" * 44,
        f"Lowest P(rr)         : {min_rr_row['P_rr']:.6f} at R = {min_rr_row['spacing_um']:.3f} um",
        f"Highest P(rr)        : {max_rr_row['P_rr']:.6f} at R = {max_rr_row['spacing_um']:.3f} um",
    ]

    right_lines = [
        "Selected spacing distribution",
        "-" * 44,
        f"Selected R           : {selected_row['spacing_um']:.3f} um",
        f"Interaction V/2π     : {selected_row['interaction_mhz']:.6f} MHz",
        f"P(gg)                : {selected_row['P_gg']:.6f}",
        f"P(gr)                : {selected_row['P_gr']:.6f}",
        f"P(rg)                : {selected_row['P_rg']:.6f}",
        f"P(rr)                : {selected_row['P_rr']:.6f}",
        f"1 - P(rr)            : {selected_row['blockade_suppression']:.6f}",
    ]

    col_width = 66
    n_rows = max(len(left_lines), len(right_lines))

    for i in range(n_rows):
        left = left_lines[i] if i < len(left_lines) else ""
        right = right_lines[i] if i < len(right_lines) else ""
        print(f"{left:<{col_width}}{right}")
        

In [42]:
# Cell 2: user-facing run parameters and UI controls

import ipywidgets as widgets
from IPython.display import display

# Fixed physics parameters
rabi_amplitude = 2.0
detuning = 0.0
duration = 1.0

# User-facing sweep and sampling controls
R_min_widget = widgets.FloatSlider(
    value=3.0,
    min=2.0,
    max=10.0,
    step=0.25,
    description="R min (um)",
    continuous_update=False,
)

R_max_widget = widgets.FloatSlider(
    value=15.0,
    min=5.0,
    max=25.0,
    step=0.5,
    description="R max (um)",
    continuous_update=False,
)

n_distances_widget = widgets.IntSlider(
    value=25,
    min=5,
    max=61,
    step=2,
    description="# distances",
    continuous_update=False,
)

selected_spacing_widget = widgets.FloatSlider(
    value=12.0,
    min=2.0,
    max=25.0,
    step=0.25,
    description="Selected R",
    continuous_update=False,
)

shots_widget = widgets.IntSlider(
    value=500,
    min=100,
    max=5000,
    step=100,
    description="Shots",
    continuous_update=False,
)

ui_controls = widgets.VBox(
    [
        widgets.HTML("<b>Bloqade blockade sweep controls</b>"),
        R_min_widget,
        R_max_widget,
        n_distances_widget,
        selected_spacing_widget,
        shots_widget,
    ]
)

display(ui_controls)

In [43]:
# Cell 3: graphics and compact interpretation

from IPython.display import clear_output

run_button = widgets.Button(
    description="Run sweep",
    button_style="primary",
)

run_output = widgets.Output()


def run_and_plot_blockade_demo(_=None):
    global spacings_um
    global shots
    global selected_spacing_um
    global results_df
    global selected_row
    global batch
    global report

    with run_output:
        clear_output(wait=True)

        R_min = float(R_min_widget.value)
        R_max = float(R_max_widget.value)
        n_distances = int(n_distances_widget.value)

        if R_max <= R_min:
            raise ValueError("R max must be larger than R min.")

        spacings_um = np.linspace(R_min, R_max, n_distances)

        shots = int(shots_widget.value)
        selected_spacing_um = float(selected_spacing_widget.value)

        results_df = run_spacing_sweep(
            spacings_um=spacings_um,
            rabi_amplitude=rabi_amplitude,
            detuning=detuning,
            duration=duration,
            shots=shots,
        )

        selected_row = select_nearest_spacing_row(
            results_df,
            selected_spacing_um=selected_spacing_um,
        )

        # Diagnostic: inspect raw Bloqade counts at the selected spacing.
        # Use the nearest spacing actually present in the sweep.
        batch, report = run_two_atom_program(
            spacing_um=selected_row["spacing_um"],
            rabi_amplitude=rabi_amplitude,
            detuning=detuning,
            duration=duration,
            shots=shots,
        )

        print("Raw Bloqade counts at selected spacing:")
        print(report.counts())

        print("\nMapped probabilities at selected spacing:")
        print(probabilities_from_report(report))

        probability_labels = ["gg", "gr", "rg", "rr"]
        probability_values = [
            selected_row["P_gg"],
            selected_row["P_gr"],
            selected_row["P_rg"],
            selected_row["P_rr"],
        ]

        fig, axes = plt.subplots(
            1,
            3,
            figsize=(12.0, 4.0),
            constrained_layout=True,
        )
        ax1, ax2, ax3 = axes

        # Plot 1: double-excitation probability
        ax1.plot(
            results_df["spacing_um"],
            results_df["P_rr"],
            marker="o",
            markersize=4,
        )
        ax1.axvline(
            selected_row["spacing_um"],
            linestyle="--",
            linewidth=1,
            alpha=0.6,
        )
        ax1.set_xlabel("Atom spacing R (um)")
        ax1.set_ylabel("P(rr)")
        ax1.set_ylim(0.0, 1.0)
        ax1.set_title("Double Excitation")
        ax1.grid(True, alpha=0.3)
        ax1.set_box_aspect(1)

        # Plot 2: Rydberg interaction strength
        ax2.plot(
            results_df["spacing_um"],
            results_df["interaction_mhz"],
            marker="o",
            markersize=4,
        )
        ax2.axvline(
            selected_row["spacing_um"],
            linestyle="--",
            linewidth=1,
            alpha=0.6,
        )
        ax2.set_xlabel(r"Atom spacing $R$ ($\mu$m)")
        ax2.set_ylabel(r"Interaction $V/2\pi$ (MHz)")
        ax2.set_title("Rydberg Interaction")
        ax2.grid(True, alpha=0.3)
        ax2.set_box_aspect(1)

        # Plot 3: final probability distribution at selected spacing
        ax3.bar(probability_labels, probability_values)
        ax3.set_xlabel("Final state")
        ax3.set_ylabel("Probability")
        ax3.set_ylim(0.0, 1.0)
        ax3.set_title(
            f"Distribution at R = {selected_row['spacing_um']:.2f} um\n"
            f"P(rr) = {selected_row['P_rr']:.3f}, "
            f"P(single) = {selected_row['P_single']:.3f}"
        )
        ax3.grid(True, axis="y", alpha=0.3)
        ax3.set_box_aspect(1)

        plt.show()

        print_compact_summary(
            df=results_df,
            selected_row=selected_row,
            rabi_amplitude=rabi_amplitude,
            detuning=detuning,
            duration=duration,
            shots=shots,
        )

        display(results_df)


run_button.on_click(run_and_plot_blockade_demo)

display(run_button, run_output)

# Run once with the default UI values
run_and_plot_blockade_demo()

Button(button_style='primary', description='Run sweep', style=ButtonStyle())

Output()